# Swin-B

In [31]:
# If running in a fresh environment, uncomment to install dependencies:
!pip -q install torch torchvision matplotlib tqdm numpy certifi Pillow pandas scikit-learn

import os

IMAGE_SIZE = 384
BATCHSIZE  = 32
NUM_WORKERS = 10
PREFETCH_FACTOR = 3
base_dir   = '/workspace/144FinalProjectDataset'
TRAIN_PATH = os.path.join(base_dir, 'train')
TEST_PATH  = os.path.join(base_dir, 'test')

print(base_dir)
print(TRAIN_PATH)
print(TEST_PATH)

import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset, Subset
import ssl
import certifi
from PIL import Image
import re

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

print('torch:', torch.__version__)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('device:', device)

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.use_deterministic_algorithms(False)
    except Exception as e:
        print('Warning:', e)

set_seed(42)

/workspace/144FinalProjectDataset
/workspace/144FinalProjectDataset/train
/workspace/144FinalProjectDataset/test
torch: 2.8.0+cu128
device: cuda


## Configuration

In [33]:
# True  → 8/2 train/val split; tracks val accuracy; saves best-val checkpoint
# False → train on ALL data (no val loop); saves final checkpoint → use for Kaggle
USE_VALIDATION = False

# No Catastrophic Loss But No Gains After Best Epoch During Val -> Keep 75
if USE_VALIDATION:
    EPOCHS = 75
else:
    EPOCHS = 75

LR_CLASSIFIER      = 1e-3
CLASSIFIER_DROPOUT = 0.4
LR_LAST_2_BLOCKS   = 1e-4
LR_ALL_FEATURES    = 2e-5
WARMUP_EPOCHS      = 5
T_0                = 10
LABEL_SMOOTHING    = 0.05
ADAMW_WEIGHT_DECAY = 1e-3
NUM_TTA_PASSES     = 5
BACKBONE_DROPOUT = 0.15
STOCHASTIC_DEPTH_PROB = 0.2
MIXUP_ALPHA = 0.2

## Loading the Data

In [34]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [35]:
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir  = test_dir
        self.transform = transform

        jpg_files = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]

        def sort_key(fname):
            m = re.findall(r'(\d+)\.jpg', fname)
            return int(m[0]) if m else float('inf')

        self.image_names = sorted(jpg_files, key=sort_key)
        print(f'TestDataset initialized: Found {len(self.image_names)} .jpg images in {test_dir}')

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image = Image.open(os.path.join(self.test_dir, image_name)).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, image_name

In [36]:
import collections

full_train_dataset             = datasets.ImageFolder(root=TRAIN_PATH, transform=train_transform)
full_train_dataset_raw_targets = datasets.ImageFolder(root=TRAIN_PATH, transform=None)

class_indices = collections.defaultdict(list)
for i, target in enumerate(full_train_dataset_raw_targets.targets):
    class_indices[target].append(i)

train_indices = []
val_indices   = []

if USE_VALIDATION:
    TRAIN_PER_CLASS = 8
    VAL_PER_CLASS   = 2
    for class_id in sorted(class_indices.keys()):
        indices = class_indices[class_id]
        random.shuffle(indices)
        folder_name = full_train_dataset_raw_targets.classes[class_id]
        if len(indices) < (TRAIN_PER_CLASS + VAL_PER_CLASS):
            print(f"Warning: '{folder_name}' has only {len(indices)} images.")
            train_indices.extend(indices[:TRAIN_PER_CLASS])
            val_indices.extend(indices[TRAIN_PER_CLASS:])
        else:
            train_indices.extend(indices[:TRAIN_PER_CLASS])
            val_indices.extend(indices[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS])
else:
    train_indices = list(range(len(full_train_dataset_raw_targets)))

train_dataset = Subset(datasets.ImageFolder(root=TRAIN_PATH, transform=train_transform), train_indices)
val_dataset   = Subset(datasets.ImageFolder(root=TRAIN_PATH, transform=val_transform), val_indices) \
                if USE_VALIDATION else None

print(f"Training samples: {len(train_dataset)}" +
      (f" | Validation samples: {len(val_indices)}" if USE_VALIDATION
       else " (full dataset, no validation)"))

Training samples: 1079 (full dataset, no validation)


### Correcting Class to Index Mapping

`datasets.ImageFolder` sorts class directories alphabetically, so '10' maps to index 2, '2' to index 12, etc. The assignment requires numerical order (class '0' → label 0, class '1' → label 1, ...). The cell below fixes this.

In [37]:
original_class_names = full_train_dataset.classes
int_class_names      = sorted([int(c) for c in original_class_names])
correct_class_to_idx = {str(n): i for i, n in enumerate(int_class_names)}
correct_idx_to_class = {v: k for k, v in correct_class_to_idx.items()}

print('Corrected class_to_idx:', correct_class_to_idx)

def remap_subset_targets(subset_dataset, original_full_dataset, new_class_to_idx):
    remapped = []
    for idx in subset_dataset.indices:
        orig_class_idx  = original_full_dataset.targets[idx]
        orig_class_name = original_full_dataset.classes[orig_class_idx]
        remapped.append(new_class_to_idx[orig_class_name])
    subset_dataset.targets = remapped
    return subset_dataset

train_dataset.dataset.class_to_idx = correct_class_to_idx
train_dataset = remap_subset_targets(train_dataset, full_train_dataset, correct_class_to_idx)
train_loader  = DataLoader(train_dataset, batch_size=BATCHSIZE, shuffle=True, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)

if USE_VALIDATION:
    val_dataset.dataset.class_to_idx = correct_class_to_idx
    val_dataset = remap_subset_targets(val_dataset, full_train_dataset, correct_class_to_idx)
    val_loader  = DataLoader(val_dataset, batch_size=BATCHSIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)
else:
    val_loader = None

print('First 10 train targets:', train_dataset.targets[:10])
if USE_VALIDATION:
    print('First 10 val targets:  ', val_dataset.targets[:10])

Corrected class_to_idx: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10, '11': 11, '12': 12, '13': 13, '14': 14, '15': 15, '16': 16, '17': 17, '18': 18, '19': 19, '20': 20, '21': 21, '22': 22, '23': 23, '24': 24, '25': 25, '26': 26, '27': 27, '28': 28, '29': 29, '30': 30, '31': 31, '32': 32, '33': 33, '34': 34, '35': 35, '36': 36, '37': 37, '38': 38, '39': 39, '40': 40, '41': 41, '42': 42, '43': 43, '44': 44, '45': 45, '46': 46, '47': 47, '48': 48, '49': 49, '50': 50, '51': 51, '52': 52, '53': 53, '54': 54, '55': 55, '56': 56, '57': 57, '58': 58, '59': 59, '60': 60, '61': 61, '62': 62, '63': 63, '64': 64, '65': 65, '66': 66, '67': 67, '68': 68, '69': 69, '70': 70, '71': 71, '72': 72, '73': 73, '74': 74, '75': 75, '76': 76, '77': 77, '78': 78, '79': 79, '80': 80, '81': 81, '82': 82, '83': 83, '84': 84, '85': 85, '86': 86, '87': 87, '88': 88, '89': 89, '90': 90, '91': 91, '92': 92, '93': 93, '94': 94, '95': 95, '96': 96, '97': 97, '98': 98, '99':

## Model Architecture

In [38]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.ops import StochasticDepth

model = models.swin_b(
    weights=models.Swin_B_Weights.IMAGENET1K_V1
)

# Set backbone neuron dropout
for m in model.modules():

    # Swin blocks use nn.Dropout internally
    if isinstance(m, nn.Dropout):
        m.p = BACKBONE_DROPOUT

# Set backbone stochastic depth
# torchvision pretrained Swin-B uses 0.5
for m in model.modules():
    if isinstance(m, StochasticDepth):
        m.p *= STOCHASTIC_DEPTH_PROB / 0.5

# Freeze everything initially
for p in model.parameters():
    p.requires_grad = False

# Replace classifier head
in_features = model.head.in_features

model.head = nn.Sequential(
    nn.Dropout(CLASSIFIER_DROPOUT),
    nn.Linear(in_features, 100)
)

model = model.to(device)

# Loss
criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)

# Sanity checks
dropouts = sorted(
    set(
        round(m.p, 3)
        for m in model.modules()
        if isinstance(m, nn.Dropout)
    )
)

stoch_depths = sorted(
    set(
        round(m.p, 3)
        for m in model.modules()
        if isinstance(m, StochasticDepth)
    )
)

print("Dropout values:", dropouts)
print("Stochastic depth values:", stoch_depths)

Dropout values: [0.15, 0.4]
Stochastic depth values: [0.0, 0.009, 0.017, 0.026, 0.035, 0.043, 0.052, 0.061, 0.07, 0.078, 0.087, 0.096, 0.104, 0.113, 0.122, 0.13, 0.139, 0.148, 0.157, 0.165, 0.174, 0.183, 0.191, 0.2]


## Training Loop

In [39]:
import numpy as np
import torch

def mixup_data(x, y, alpha=0.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)

    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]

    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam

In [40]:
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return (
        lam * criterion(pred, y_a)
        + (1 - lam) * criterion(pred, y_b)
    )

In [41]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=MIXUP_ALPHA)
        
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(images)
            # loss    = criterion(outputs, labels)
            loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

    return running_loss / total, correct / total

In [42]:
import torch.optim as optim

# Unfreeze the entire network
for param in model.features.parameters():
    param.requires_grad = True

# Per-group learning rates: head > last_2_blocks > rest of features
head_params         = list(model.head.parameters())
head_ids            = set(id(p) for p in head_params)

last_2_block_params = [p for p in model.features[-2].parameters() if id(p) not in head_ids]
assigned_ids        = head_ids | set(id(p) for p in last_2_block_params)

rest_params = [p for p in model.features.parameters() if id(p) not in assigned_ids]

optimizer = optim.AdamW([
    {'params': head_params,         'lr': LR_CLASSIFIER},
    {'params': last_2_block_params, 'lr': LR_LAST_2_BLOCKS},
    {'params': rest_params,         'lr': LR_ALL_FEATURES},
], weight_decay=ADAMW_WEIGHT_DECAY)

# warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
#     optimizer, start_factor=1e-3, end_factor=1.0, total_iters=WARMUP_EPOCHS
# )
# cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     optimizer, T_0=T_0, T_mult=1, eta_min=1e-6
# )
# scheduler = torch.optim.lr_scheduler.SequentialLR(
#     optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS]
# )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.000001)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

Trainable params: 86,843,676


In [43]:
os.makedirs('./checkpoints', exist_ok=True)
ckpt_path = './checkpoints/swinb_best.pth'

history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)

    if USE_VALIDATION:
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        scheduler.step()
        print(f'  Ep {epoch:03d}/{EPOCHS} | train {train_loss:.4f}/{train_acc:.4f} | val {val_loss:.4f}/{val_acc:.4f}')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch}, ckpt_path)
            print(f'    --> Saved best: {best_val_acc*100:.2f}%')
    else:
        scheduler.step()
        print(f'  Ep {epoch:03d}/{EPOCHS} | train {train_loss:.4f}/{train_acc:.4f}')

if not USE_VALIDATION:
    torch.save({'model_state_dict': model.state_dict(), 'epoch': EPOCHS}, ckpt_path)
    print(f'\nFull-dataset training complete. Saved to {ckpt_path}')
else:
    print(f'\nBest val acc: {best_val_acc*100:.2f}%')

  Ep 001/75 | train 3.9319/0.0575
  Ep 002/75 | train 2.6192/0.2595
  Ep 003/75 | train 2.0159/0.3355
  Ep 004/75 | train 1.8892/0.4365
  Ep 005/75 | train 1.5885/0.4152
  Ep 006/75 | train 1.4295/0.3939
  Ep 007/75 | train 1.2576/0.3957
  Ep 008/75 | train 1.2186/0.5144
  Ep 009/75 | train 1.2733/0.3327
  Ep 010/75 | train 0.9869/0.4087
  Ep 011/75 | train 1.1657/0.3513
  Ep 012/75 | train 1.2041/0.4143
  Ep 013/75 | train 0.9129/0.4588
  Ep 014/75 | train 1.0638/0.5032
  Ep 015/75 | train 0.8993/0.5690
  Ep 016/75 | train 1.1475/0.6080
  Ep 017/75 | train 0.8311/0.4875
  Ep 018/75 | train 0.9626/0.5607
  Ep 019/75 | train 1.0661/0.4449
  Ep 020/75 | train 0.9666/0.4272
  Ep 021/75 | train 1.0617/0.5218
  Ep 022/75 | train 0.9699/0.6608
  Ep 023/75 | train 0.9378/0.4291
  Ep 024/75 | train 0.8827/0.4597
  Ep 025/75 | train 0.9224/0.4717
  Ep 026/75 | train 0.9617/0.5551
  Ep 027/75 | train 0.8694/0.5690
  Ep 028/75 | train 0.9912/0.4708
  Ep 029/75 | train 0.8966/0.4866
  Ep 030/75 | 

### Train/Val Curves

In [44]:
if USE_VALIDATION:
    epochs_range = range(1, len(history['train_loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
    ax1.plot(epochs_range, history['val_loss'],   label='Val Loss')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()

    ax2.plot(epochs_range, history['train_acc'], label='Train Acc')
    ax2.plot(epochs_range, history['val_acc'],   label='Val Acc')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print('Skipping Train/Val Curves (USE_VALIDATION=False).')

Skipping Train/Val Curves (USE_VALIDATION=False).


### Generate Submission (No TTA)

Single forward pass per test image.

In [45]:
import pandas as pd

test_transform_base = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset_no_tta = TestDataset(TEST_PATH, transform=test_transform_base)
test_loader_no_tta  = DataLoader(test_dataset_no_tta, batch_size=BATCHSIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)

loaded_checkpoint = torch.load(ckpt_path)
model.load_state_dict(loaded_checkpoint['model_state_dict'])
model.to(device)

model.eval()
image_ids_no_tta = []
predicted_labels = []

with torch.no_grad():
    for images, names in tqdm(test_loader_no_tta, desc='Generating Submission (No TTA)'):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        image_ids_no_tta.extend(names)
        predicted_labels.extend(preds.cpu().numpy())

idx_to_correct_label = {v: int(k) for k, v in full_train_dataset.class_to_idx.items()}
preds_no_tta = [idx_to_correct_label[p] for p in predicted_labels]

df_no_tta = pd.DataFrame({'ID': image_ids_no_tta, 'Label': preds_no_tta})
df_no_tta['ID_num'] = df_no_tta['ID'].apply(lambda x: int(os.path.splitext(x)[0]))
df_no_tta = df_no_tta.sort_values('ID_num').drop(columns=['ID_num']).reset_index(drop=True)
df_no_tta.to_csv('SWINsubmission_no_tta.csv', index=False)

print('SWINsubmission_no_tta.csv created!')
print(df_no_tta.head())

TestDataset initialized: Found 1036 .jpg images in /workspace/144FinalProjectDataset/test


Generating Submission (No TTA):   0%|          | 0/33 [00:00<?, ?it/s]

SWINsubmission_no_tta.csv created!
      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     62
4  4.jpg     42


### Test-Time Augmentation (TTA)

Multiple augmented passes per image averaged together.

In [46]:
tta_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if USE_VALIDATION:
    val_dataset_tta        = datasets.ImageFolder(root=TRAIN_PATH, transform=tta_transform)
    val_dataset_tta_subset = Subset(val_dataset_tta, val_dataset.indices)
    val_loader_tta         = DataLoader(val_dataset_tta_subset, batch_size=BATCHSIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)
    print('Created val_loader_tta for Test-Time Augmentation.')
else:
    print('Skipping TTA validation (USE_VALIDATION=False).')

Skipping TTA validation (USE_VALIDATION=False).


In [47]:
if USE_VALIDATION:
    loaded_checkpoint = torch.load(ckpt_path)
    model.load_state_dict(loaded_checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    # Use the corrected ground-truth labels directly (DataLoader labels are un-remapped)
    all_labels = torch.tensor(val_dataset.targets)
    all_probs  = torch.zeros(len(val_dataset), 100)

    # Each iteration of val_loader_tta applies fresh random augmentations
    with torch.no_grad():
        for pass_idx in range(NUM_TTA_PASSES):
            pass_probs = []
            for images, _ in tqdm(val_loader_tta, desc=f'TTA pass {pass_idx+1}/{NUM_TTA_PASSES}'):
                images = images.to(device)
                probs  = torch.softmax(model(images), dim=1)
                pass_probs.append(probs.cpu())
            all_probs += torch.cat(pass_probs, dim=0)

    avg_probs    = all_probs / NUM_TTA_PASSES
    _, predicted = torch.max(avg_probs, 1)
    tta_accuracy = (predicted == all_labels).sum().item() / len(all_labels)
    print(f'Validation Accuracy with TTA ({NUM_TTA_PASSES} passes): {tta_accuracy * 100:.2f}%')
else:
    print('Skipping TTA validation (USE_VALIDATION=False).')

Skipping TTA validation (USE_VALIDATION=False).


### Submitting To Kaggle (TTA)

Testing the model with TTA individually.

In [48]:
test_dataset_tta = TestDataset(TEST_PATH, transform=tta_transform)
test_loader_tta  = DataLoader(test_dataset_tta, batch_size=BATCHSIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)

loaded_checkpoint = torch.load(ckpt_path)
model.load_state_dict(loaded_checkpoint['model_state_dict'])
model.to(device)
model.eval()

image_ids_tta = []
all_probs     = None

with torch.no_grad():
    for pass_idx in range(NUM_TTA_PASSES):
        pass_probs = []
        for images, names in tqdm(test_loader_tta, desc=f'TTA pass {pass_idx+1}/{NUM_TTA_PASSES}'):
            images = images.to(device)
            probs  = torch.softmax(model(images), dim=1)
            pass_probs.append(probs.cpu())
            if pass_idx == 0:
                image_ids_tta.extend(names)
        pass_probs = torch.cat(pass_probs, dim=0)
        all_probs  = pass_probs if all_probs is None else all_probs + pass_probs

avg_probs = all_probs / NUM_TTA_PASSES
_, preds  = torch.max(avg_probs, 1)

idx_to_correct_label = {v: int(k) for k, v in full_train_dataset.class_to_idx.items()}
preds_tta = [idx_to_correct_label[p] for p in preds.numpy()]

df_tta = pd.DataFrame({'ID': image_ids_tta, 'Label': preds_tta})
df_tta['ID_num'] = df_tta['ID'].apply(lambda x: int(os.path.splitext(x)[0]))
df_tta = df_tta.sort_values('ID_num').drop(columns=['ID_num']).reset_index(drop=True)
df_tta.to_csv('SWINsubmission.csv', index=False)

print('SWINsubmission.csv created!')
print(df_tta.head())

TestDataset initialized: Found 1036 .jpg images in /workspace/144FinalProjectDataset/test


TTA pass 1/5:   0%|          | 0/33 [00:00<?, ?it/s]

TTA pass 2/5:   0%|          | 0/33 [00:00<?, ?it/s]

TTA pass 3/5:   0%|          | 0/33 [00:00<?, ?it/s]

TTA pass 4/5:   0%|          | 0/33 [00:00<?, ?it/s]

TTA pass 5/5:   0%|          | 0/33 [00:00<?, ?it/s]

SWINsubmission.csv created!
      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     62
4  4.jpg     42
